In [1]:
import os
import cv2
import numpy as np
from PIL import Image
from pdf2image import convert_from_path

import torch
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    AutoProcessor,
    Qwen2VLForConditionalGeneration
)

# ----------------------------
# PATHS
# ----------------------------
pdf_path = r"C:\Users\pardh\Downloads\PDP\24-25 Assignment 1\Please upload your assignment file (in .pdf format) (File responses)\22BCS001 - ABHIGYAN NIRANJAN IIIT Dharwad.pdf"

poppler_path = r"C:\Users\pardh\Downloads\PDP\Release-26.02.0-0\poppler-26.02.0\Library\bin"

output_path = r"C:\Users\pardh\Downloads\PDP\hybrid_output.txt"

# ----------------------------
# LOAD MODELS
# ----------------------------
print("Loading TrOCR...")
trocr_processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
trocr_model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

print("Loading Qwen2-VL...")
qwen_processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-0.5B-Instruct")
qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-0.5B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)

# ----------------------------
# OCR FUNCTIONS
# ----------------------------
def trocr_extract_lines(page):
    img = np.array(page)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    _, thresh = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY_INV)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 5))
    dilated = cv2.dilate(thresh, kernel, iterations=1)

    contours, _ = cv2.findContours(
        dilated,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if h > 15 and w > 50:
            boxes.append((x, y, w, h))

    boxes = sorted(boxes, key=lambda b: b[1])

    page_text = ""

    for x, y, w, h in boxes:
        line = img[y:y+h, x:x+w]
        line_img = Image.fromarray(line).convert("RGB")

        pixel_values = trocr_processor(
            images=line_img,
            return_tensors="pt"
        ).pixel_values

        generated_ids = trocr_model.generate(
            pixel_values,
            max_new_tokens=150
        )

        text = trocr_processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )[0].strip()

        if len(text) > 1:
            page_text += text + "\n"

    return page_text


def qwen_fallback(page, page_no):
    img_path = f"temp_page_{page_no}.jpg"
    page.save(img_path)

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": img_path
                },
                {
                    "type": "text",
                    "text": "Extract all text exactly from this document page. Preserve formulas, symbols, layout and equations."
                }
            ]
        }
    ]

    text = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs = qwen_processor(
        text=[text],
        images=[Image.open(img_path)],
        padding=True,
        return_tensors="pt"
    )

    image_inputs = image_inputs.to(qwen_model.device)

    generated_ids = qwen_model.generate(
        **image_inputs,
        max_new_tokens=2048
    )

    output = qwen_processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    return output

# ----------------------------
# PROCESS PDF
# ----------------------------
pages = convert_from_path(
    pdf_path,
    dpi=300,
    poppler_path=poppler_path
)

final_text = ""

for i, page in enumerate(pages):
    print(f"Processing page {i+1}")

    trocr_text = trocr_extract_lines(page)

    print("TrOCR chars:", len(trocr_text))

    if len(trocr_text.strip()) < 200:
        print("Using Qwen fallback")
        page_text = qwen_fallback(page, i)
    else:
        page_text = trocr_text

    final_text += f"\n========== PAGE {i+1} ==========\n"
    final_text += page_text + "\n"

# ----------------------------
# SAVE
# ----------------------------
with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_text)

print(final_text)
print("saved")

C:\Users\pardh\Downloads\PDP\pdp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading TrOCR...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 478/478 [00:00<00:00, 7194.42it/s]
[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading Qwen2-VL...


OSError: Qwen/Qwen2-VL-0.5B-Instruct is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`